# N5_Working_Capital_Levers
## Treasury Module

**CFOPackV001: Treasury Decision Workshop**

---

© Copyright 2026 Professor Vinaya Sathyanarayana

All rights reserved. This notebook is provided as part of the CFOPackV001 Treasury Decision Workshop.
Attribution required: Please retain this copyright notice and credit Professor Vinaya Sathyanarayana in any derivative work.

**Contact:** vinallcontact@gmail.com  
**GitHub:** https://github.com/VinayaSharada/KateelLearningDemosToStudents


## Learning Objectives

By the end of this module, you will be able to:

- **Understand** the business decision this notebook supports
- **Execute** the analysis workflow without errors
- **Interpret** outputs in plain English
- **Explain** the assumptions behind each calculation
- **Adapt** the code for your own data

**Estimated Time:** ** 20-40 minutes (including reading code and outputs)


## Overview### What This Notebook DoesThis notebook models the impact of working capital interventions—accelerating collections, negotiating extended payables, and optimizing inventory—to improve cash position.### Why It MattersWorking capital management can dramatically improve cash without external funding:- **Collections lever:** What if you could reduce payment delays by 5 or 10 days?- **Payables lever:** What if you negotiated 30-day terms instead of 15 days?- **Inventory lever:** What if you reduced inventory holding by 10 or 20%?- **Scenario comparison:** See cost-benefit of each intervention### What Data It Uses- `N4_revised_forecast.csv` – Baseline realistic forecast- Customer Payment Cycle (CCC) assumptions (DSO, DPO, DIO)### What Outputs It Creates- `N5_ccc_scenarios.csv` – Cash position under different working capital scenarios- `N5_lever_comparison.csv` – Impact summary of each intervention

## Execute Workflow

# N5_Working_Capital_Levers
## Treasury Module

**CFOPackV001: Treasury Decision Workshop**

---

© Copyright 2026 Professor Vinaya Sathyanarayana

All rights reserved. This notebook is provided as part of the CFOPackV001 Treasury Decision Workshop.
Attribution required: Please retain this copyright notice and credit Professor Vinaya Sathyanarayana in any derivative work.

**Contact:** vinallcontact@gmail.com  
**GitHub:** https://github.com/VinayaSharada/KateelLearningDemosToStudents


## Learning Objectives

By the end of this module, you will be able to:

- **Understand** the business decision this notebook supports
- **Execute** the analysis workflow without errors
- **Interpret** outputs in plain English
- **Explain** the assumptions behind each calculation
- **Adapt** the code for your own data

**Estimated Time:** ** 20-40 minutes (including reading code and outputs)


In [ ]:
# ==============================================================================
# SETUP: Imports and Configuration
# ==============================================================================
# This cell imports all required libraries and configures data sources.
# No changes needed unless you want to use your own data.

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
# CONFIGURATION: Choose your data source
USE_GITHUB_DATA = True  # Set to False if you want to upload your own data
GITHUB_RAW_URL = 'https://raw.githubusercontent.com/VinayaSharada/KateelLearningDemosToStudents/main/CFOPackV001/data/synthetic'

print('✓ Imports successful')
print(f"✓ Data source: {'GitHub (synthetic)' if USE_GITHUB_DATA else 'Manual upload'}")

In [ ]:
# ==============================================================================
# LOAD DATA FROM PREVIOUS NOTEBOOKS (N1, N2, N3, N4)
# ==============================================================================


print("[[LOAD] Loading data from previous notebooks...")
print()
# Load validated data and predictions
try:
    validated_data = pd.read_csv("../outputs/N1_validated_data.csv")
print(f"[OK] Loaded validated data from N1: {len(validated_data)} invoices")
except FileNotFoundError:
    print("[WARNING] N1 validated data not found")
    validated_data = None

try:
    predictions = pd.read_csv("../outputs/N3_invoice_payment_predictions.csv")
print(f"[OK] Loaded predictions from N3: {len(predictions)} invoices")
except FileNotFoundError:
    print("[WARNING] N3 predictions not found")
    predictions = None

try:
    gap_analysis = pd.read_csv("../outputs/N4_gap_analysis.csv")
print(f"[OK] Loaded gap analysis from N4: {len(gap_analysis)} days")
except FileNotFoundError:
    print("[WARNING] N4 gap analysis not found")
    gap_analysis = None

# Calculate target gap (cash to be closed)
if gap_analysis is not None:
    target_gap = gap_analysis['gap'].sum()
print(f"[OK] Target gap identified: ${target_gap:,.0f}")
else:
    target_gap = 500000  # Default assumption
print(f"[NOTE] Using default target gap: ${target_gap:,.0f}")
print()
# ============================================================================

In [ ]:
# ==============================================================================
# DATA LOADING: GitHub or Manual Upload
# ==============================================================================
# Define two data loading methods and use whichever matches your choice above.

def load_data_from_github():
    """Load synthetic data directly from GitHub repository.
    
    Advantages:
    - No API key required
    - Pre-validated and consistent with reference outputs
    - Fast (uses GitHub CDN)
    
    Returns: dict with keys 'invoices', 'payments', 'customers'
    """
    try:
        print('Loading data from GitHub...')
        invoices = pd.read_csv(f'{GITHUB_RAW_URL}/invoices.csv')
        payments = pd.read_csv(f'{GITHUB_RAW_URL}/payments.csv')
        customers = pd.read_csv(f'{GITHUB_RAW_URL}/customers.csv')
        
        print(f'✓ Loaded {len(invoices):,} invoices')
        print(f'✓ Loaded {len(payments):,} payments')
        print(f'✓ Loaded {len(customers):,} customers')
        return {'invoices': invoices, 'payments': payments, 'customers': customers}
    except Exception as e:
        print(f'✗ Error: {e}')
        print('  Try Option 2: Manual upload')
        return None

def load_data_from_upload():
    """Load data from files you upload manually.
    
    In Colab: Click Files panel → Upload → Select CSVs
    In Jupyter: Put CSVs in the same folder as this notebook
    
    Required files: invoices.csv, payments.csv, customers.csv
    See data/README.md for required columns.
    """
    try:
        print('Loading data from uploaded files...')
        invoices = pd.read_csv('invoices.csv')
        payments = pd.read_csv('payments.csv')
        customers = pd.read_csv('customers.csv')
        
        print(f'✓ Loaded {len(invoices):,} invoices')
        print(f'✓ Loaded {len(payments):,} payments')
        print(f'✓ Loaded {len(customers):,} customers')
        return {'invoices': invoices, 'payments': payments, 'customers': customers}
    except FileNotFoundError as e:
        print(f'✗ File not found: {e}')
        return None

# Execute data loading based on configuration above
if USE_GITHUB_DATA:
    data = load_data_from_github()
else:
    data = load_data_from_upload()

if data is None:
    print('\n⚠ Data loading failed. Check error above.')
else:
    invoices = data['invoices']
    payments = data['payments']
    customers = data['customers']
    print('\n✓ All data loaded and ready for analysis!')


## CURRENT STATE METRICS

**Purpose:** Calculate baseline metrics for working capital assessment.

**Code Section:** ~18 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[ Current Working Capital Metrics")
print()
# Calculate baseline CCC metrics (from customer data)total_ar = predictions['amount_usd'].sum()avg_payment_terms = validated_data['payment_terms_days'].mean()avg_dso = predictions['predicted_days_late'].mean() + avg_payment_terms
# Assumptions for DIO and DPO (would come from accounting system)assumed_inventory_value = 8_000_000assumed_payables_value = 10_000_000assumed_cogs = 60_000_000dio = (assumed_inventory_value / assumed_cogs) * 365dpo = (assumed_payables_value / assumed_cogs) * 365ccc = avg_dso + dio - dpoprint(f"  DSO (Days Sales Outstanding):        {avg_dso:.1f} days")
print(f"  DIO (Days Inventory Outstanding):    {dio:.1f} days")
print(f"  DPO (Days Payable Outstanding):      {dpo:.1f} days")
print(f"  CCC (Cash Conversion Cycle):         {ccc:.1f} days")
print()

## SCENARIO MODELING

**Purpose:** Model different working capital management scenarios.

**Code Section:** ~85 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[GOAL] SCENARIO ANALYSIS")
print()scenarios = []
# Scenario 0: Baseline (no changes)scenarios.append({    'scenario': 'Baseline',    'dso_reduction': 0,    'dio_reduction': 0,    'dpo_increase': 0,    'description': 'No changes'})
# Scenario 1: Aggressive Collections (reduce DSO 5 days)dso_reduction = 5cash_impact
_1 = (dso_reduction / 365) * total_arscenarios.append({    'scenario': 'Collections Push',    'dso_reduction': dso_reduction,    'dio_reduction': 0,    'dpo_increase': 0,    'description': 'Activate dunning + early-pay discounts',    'cash_impact': cash_impact_1})
# Scenario 2: Inventory Reduction (reduce DIO 10%)dio_reduction_pct = 10cash_impact
_2 = (dio_reduction_pct / 100) * assumed_inventory_valuescenarios.append({    'scenario': 'Inventory Reduction',    'dso_reduction': 0,    'dio_reduction': dio_reduction_pct,    'dpo_increase': 0,    'description': 'JIT inventory + demand forecast',    'cash_impact': cash_impact_2})
# Scenario 3: Extend Payables (increase DPO 7 days)dpo_increase = 7cash_impact
_3 = (dpo_increase / 365) * assumed_cogsscenarios.append({    'scenario': 'Extend Payables',    'dso_reduction': 0,    'dio_reduction': 0,    'dpo_increase': dpo_increase,    'description': 'Negotiate extended terms with suppliers',    'cash_impact': cash_impact_3})
# Scenario 4: Combined (Collections + Payables)cash_impact
_4 = cash_impact_1 + cash_impact_3scenarios.append({    'scenario': 'Combined (Collections + Payables)',    'dso_reduction': dso_reduction,    'dio_reduction': 0,    'dpo_increase': dpo_increase,    'description': 'Both collections push AND payables extension',    'cash_impact': cash_impact_4})
# Scenario 5: Aggressive Combinedcash_impact
_5 = cash_impact_1 + cash_impact_2 + cash_impact_3scenarios.append({    'scenario': 'All Three Levers',    'dso_reduction': dso_reduction,    'dio_reduction': dio_reduction_pct,    'dpo_increase': dpo_increase,    'description': 'Collections + Inventory + Payables',    'cash_impact': cash_impact_5})
# Convert to dataframescenarios_df = pd.DataFrame(scenarios)
print("[Scenario Summary:")
print("[-" * 100)for idx, row in scenarios_df.iterrows():    if 'cash_impact' in row and pd.notna(row['cash_impact']):        impact = row['cash_impact']        pct_closed = (impact / target_gap * 100) if target_gap > 0 else 0        status = "[OK] CLOSES GAP" if impact >= target_gap else f"{pct_closed:.0f}% closes gap"        print(f"{row['scenario']:30s}")
print(f"  {row['description']}")
print(f"  Cash impact: ${impact:,.0f}  ({status})")        if row['dso_reduction'] > 0:            print(f"     DSO reduction: {row['dso_reduction']:.0f} days")        if row['dio_reduction'] > 0:            print(f"     DIO reduction: {row['dio_reduction']:.0f}%")        if row['dpo_increase'] > 0:            print(f"     DPO increase: {row['dpo_increase']:.0f} days")
print()
print("[-" * 100)
print()

## RISK ASSESSMENT BY LEVER

**Purpose:** Identify and assess payment risks across customer segments.

**Code Section:** ~41 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[WARNING]  RISK ANALYSIS")
print()risks = {    'Collections': {        'impact': cash_impact_1,        'risks': [            'Customer churn from aggressive dunning',            'Early-pay discounts reduce margin',            'Sales team pushback on key accounts'        ],        'timeline': '1-2 weeks',        'feasibility': 'Medium'    },    'Inventory': {        'impact': cash_impact_2,        'risks': [            'Stockout risk if demand increases',            'Supply chain disruption impact',            'Requires operations coordination'        ],        'timeline': '4-8 weeks',        'feasibility': 'Hard'    },    'Payables': {        'impact': cash_impact_3,        'risks': [            'Supplier relationship tension',            'Lose early-pay discounts',            'May lose preferred status'        ],        'timeline': '2-3 weeks',        'feasibility': 'Medium'    }}for lever, details in risks.items():    print(f"{lever} (${details['impact']:,.0f} impact):")
print(f"  Timeline: {details['timeline']}")
print(f"  Feasibility: {details['feasibility']}")    for risk in details['risks']:        print(f"   {risk}")
print()

## RECOMMENDATION

**Purpose:** Execute recommendation analysis.

**Code Section:** ~14 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[IDEA] RECOMMENDATION")
print()
print("[Based on impact, timeline, and feasibility:")
print()
print(f"[OK] BEST APPROACH: Collections + Payables (Scenario 4)")
print(f"  Impact: ${cash_impact_4:,.0f}")
print(f"  Closes {(cash_impact_4/target_gap*100):.0f}% of gap")
print()
print("[  Why this approach:")
print("[  1. Collections is fast (1-2 weeks) and impactful")
print("[  2. Payables is negotiation-based (medium-term)")
print("[  3. Both independent (can do in parallel)")
print("[  4. If one stalls, you still have the other")
print()

## EXPORT SCENARIOS

**Purpose:** Model different working capital management scenarios.

**Code Section:** ~6 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[[SAVE] Exporting scenarios...")export_path = "../outputs/N5_ccc_scenarios.csv"os.makedirs(os.path.dirname(export_path), exist_ok=True)scenarios_df.to_csv(export_path, index=False)
print(f"[OK] Exported: {export_path}")
print()

## KEY INSIGHTS

**Purpose:** Execute key insights analysis.

**Code Section:** ~12 lines
**Estimated Time:** ** 2-3 minutes

In [ ]:
print("[=" * 80)
print("[[DONE] N5 COMPLETE - Working Capital Levers Modeled")
print("[=" * 80)
print()
print("[[INFO] Key Insights:")
print(f"   ${target_gap:,.0f} gap needs to be closed")
print(f"   Collections alone can close {(cash_impact_1/target_gap*100):.0f}% of gap")
print(f"   Payables alone can close {(cash_impact_3/target_gap*100):.0f}% of gap")
print(f"   Combined can close {(cash_impact_4/target_gap*100):.0f}% of gap")
print()
print("[[GOAL] Next step: N6_FX_Hedge_Decision.py")
print("[   Consider FX hedging strategy for open exposures")

## Download Your ResultsThis notebook generated the following files. Download them to your computer:### Output Files| File Name | Description | Size | Download ||-----------|-------------|------|----------|| N5_ccc_scenarios.csv | Cash position under different working capital scenarios | ~15 KB | [Download](#) || N5_lever_comparison.csv | Impact comparison of collections, payables, inventory levers | ~5 KB | [Download](#) |### How to Download in Colab1. Click the **Files** icon (📁) in left sidebar2. Right-click the output files folder3. Select **Download**### Where Files Are Saved- **Colab:** `/content/outputs/` (download to your computer)- **Local Jupyter:** `../outputs/` (same directory as notebook)- **Next Step:** Use these files in the next notebook### What Each File Contains- **N5_ccc_scenarios.csv:** Cash position under different working capital scenarios- **N5_lever_comparison.csv:** Impact comparison of collections, payables, inventory levers

---

## Module Complete!

You have successfully completed this module. Your outputs are ready for the next step.

**Next Module:** Open the next notebook to continue the workshop.

**Questions or Issues?**
- Review the Learning Objectives and inline comments above
- Check `participant/GETTING_STARTED.md` for help
- Email: vinallcontact@gmail.com

---
© 2026 Professor Vinaya Sathyanarayana | CFOPackV001 Treasury Decision Workshop
